# Compare Search Approaches

This notebook compares different search approaches with PyArabic integration.

In [1]:
import os
import sys
import time

sys.path.append('/app')
os.chdir('/app')

from src.utils import ServiceDatasetLoader
from src.processing import TextNormalizer, MorphReducer, LexicalRelevanceFilter
from src.search import ServiceSearch
from src.presets import get_config, list_presets

print("✅ Ready to compare approaches")

✅ Ready to compare approaches


In [2]:
# Show available presets
presets = list_presets()
print("📋 Available presets:")
for name, desc in presets.items():
    print(f"   {name}: {desc}")

📋 Available presets:
   original: Original approach: GATE-AraBert + HNSW + lexical filtering
   reranker: Pure reranker: Jina embeddings + cosine similarity + Arabic reranker
   jina_reranker: Pure Jina + Arabic reranker (might be the ultimate approach)
   hybrid: Ultimate hybrid: Jina + reranker + lexical filtering
   jina_simple: Simple Jina: jina-embeddings-v3 + FAISS indexing (baseline)
   faiss_scaled: FAISS: Jina embeddings + FAISS indexing (for larger datasets)
   arabic_focused: Arabic-focused: GATE-AraBert + reranker for Arabic, Jina for English


In [3]:
# Test friend's recommendation
config = get_config('reranker')
print(f"Testing: {config['description']}")

# Initialize components
normalizer = TextNormalizer()
morpher = MorphReducer()
lexical_filter = LexicalRelevanceFilter(normalizer)

# Load data
rename_map = {
    'الاسم عربي': 'service',
    'التصنيف عربي': 'classification', 
    'القطاع عربي': 'sector',
    'الوصف المختصر عربي': 'description_short',
    'الوصف عربي': 'description',
    'المستفيدين من الخدمة': 'beneficiaries'
}
combine_cols = ('service', 'service', 'service', 'classification', 'sector', 'description_short', 'description', 'beneficiaries')
loader_ar = ServiceDatasetLoader('data/NaamaServiceIn full Details.xlsx', rename_map, combine_cols)

print(f"Loaded {len(loader_ar.documents)} documents")

Testing: Pure reranker: Jina embeddings + cosine similarity + Arabic reranker
Loaded 997 documents


In [4]:
# Initialize search engine
engine = ServiceSearch(
    {'ar': loader_ar},
    config,
    normalizer=normalizer,
    morpher=morpher,
    lexical_filter=lexical_filter
)

print("✅ Search engine initialized")

✅ Search engine initialized


In [5]:
# Test queries
test_queries = ['فاكهة', 'احفر بير', 'دجاج', 'نحل']

for query in test_queries:
    start = time.time()
    result = engine.search(query)
    elapsed = time.time() - start
    
    hits = len(result['hits_kept'])
    print(f"\n'{query}': {hits} hits in {elapsed:.2f}s")
    
    for i, hit in enumerate(result['hits_kept'][:3], 1):
        print(f"   {i}. {hit['final_pct']:.1f}% - {hit['title'][:50]}...")


'فاكهة': 20 hits in 12.64s
   1. 62.0% - نقل ملكية ترخيص تشغيلي تربية وإنتاج جدات أمهات الد...
   2. 61.8% - نقل ملكية ترخيص تشغيلي تربية وإنتاج جدات أمهات الد...
   3. 60.4% - إلغاء شهادة اعتماد مصدر مائي (عقد توريد) لمصانع إن...

'احفر بير': 20 hits in 4.90s
   1. 60.3% - الاستيراد والتصدير للأسمدة...
   2. 58.3% - مزاولة مهنة بيطرية افراد...
   3. 56.6% - تركيب البيوت الزراعية...

'دجاج': 20 hits in 4.95s
   1. 87.9% - طلب اذن استيراد طيور حية...
   2. 87.7% - تجديد ترخيص تشغيلي تربية السمان (الفري)...
   3. 63.7% - مزاولة مهنة بيطرية افراد...

'نحل': 20 hits in 4.54s
   1. 88.9% - إذن استيراد النحل و ملكات النحل...
   2. 88.7% - إلغاء ترخيص منحل أفراد...
   3. 88.4% - نقل منحل أفراد...


## Results

This notebook demonstrates the PyArabic-integrated search system working correctly with the friend's recommended approach.